# 1. Data Loading and Exploration

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

sns.set(
    { "figure.figsize": (8, 6) },
    style='ticks',
    color_codes=True,
    font_scale=0.8
)
%config InlineBackend.figure_formats = set(('retina', 'svg'))

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.model_selection import GridSearchCV, ParameterGrid

from sklearn.preprocessing import MinMaxScaler

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

from statsmodels.formula.api import ols

from sklearn.metrics import mean_squared_error
from functools import partial
rmse = partial(mean_squared_error, squared=False)

In [ ]:
cars=pd.read_csv('adverts.csv')

#Let us inspect the data by using head()
cars.head()

In [ ]:
cars.info()

In [ ]:
cars.columns

In [ ]:
cars.shape

In [ ]:
cars.isnull().sum()

## 1.1 Visualization of noise/outliers

In [ ]:
import plotly.express as px
fig = px.scatter(cars, x="price", title="Distribution of Price")
fig.update_layout(xaxis_title="Price", yaxis_title="Index")
fig.show()

In [ ]:
import plotly.graph_objs as go
fig = go.Figure()
fig.add_trace(go.Box(x=cars['mileage'], name="Mileage"))
fig.update_layout(title="Mileage Outliers and Noise")
fig.show()

In [ ]:
def find_outliers_IQR(df):
    q1=df.quantile(0.25)
    q3=df.quantile(0.75)
    IQR=q3-q1
    outliers = df[((df<(q1-1.5*IQR)) | (df>(q3+1.5*IQR)))] 
    return outliers

#Defination for calculatng outliers using IQR method

In [ ]:
outliers = find_outliers_IQR(cars['mileage'])
print('Outliers info on Mileage')

print("Number of outliers:"+ str(len(outliers)))

print("Min outlier value:"+ str(outliers.min()))

print("Max outlier value:"+ str(outliers.max()))

##  Data Processing for Machine Learning

## 1.2 Filling Null values using pipeline

In [ ]:
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

columns_to_impute = ['standard_colour', 'body_type', 'fuel_type']

for col in columns_to_impute:
    cars[col] = pipeline.fit_transform(cars[[col]])

Another work around way to fill null values with respect to each probability

In [ ]:
colors = ['Black','White','Grey','Blue','Silver']
cars['standard_colour'].fillna(value=pd.Series(np.random.choice(colors, size=len(cars), p=[1/5,1/5,1/5,1/5,1/5])),inplace=True)

In [ ]:
body_type = ['Hatchback','SUV']
cars['body_type'].fillna(value=pd.Series(np.random.choice(body_type, size=len(cars), p=[1/2,1/2])),inplace=True)

In [ ]:
fuel_type = ['Petrol','Diesel']
cars['fuel_type'].fillna(value=pd.Series(np.random.choice(fuel_type, size=len(cars), p=[1/2,1/2])),inplace=True)

## 1.3 Dealing with Noise/Outliers

In [ ]:
cars['mileage'].isnull().sum()

In [ ]:
cars.query('mileage.isnull()').head()

In [ ]:
cars.query('mileage.isnull()')['vehicle_condition'].value_counts()

On inspection we came to know there are only 127 null values in Mileage column out of 402005 
rows. And on further investigation all these 127 cars are marked as USED in vehicle_condition. 
As it irrelevant to have null values in mileage for a used car so dropping these 127 rows from the 
dataset. Removing these rows will not affect the data since it represents only 0.03% on total 
dataset.

In [ ]:
cars = cars.dropna(subset=["mileage"])

In [ ]:
cars.query('126483<=mileage<=999999').head()

In [ ]:
cars.query('mileage>0 & reg_code.isnull() & year_of_registration.isnull() & vehicle_condition=="NEW"')

In [ ]:
outliers=cars.query('126483<=mileage<=999999')
cars.drop(outliers.index,inplace=True)

noise=cars.query('mileage==0 & vehicle_condition=="USED"')
cars.drop(noise.index,inplace=True)

noise=cars.query('mileage>0 & reg_code.isnull() & year_of_registration.isnull()')
cars.drop(noise.index,inplace=True)

noise_mileage: removing the noise data as vehicle is used and mileage is 0. If these values are filled with mean then it can effect the data as mean distribution of mileage is huge

In [ ]:
condition = (cars['vehicle_condition'] == 'NEW') & (cars['mileage'] > 0)
cars.loc[condition, 'mileage'] = 0   

As we know when a car condition is NEW, mileage should not be greater than 0. So, replacing 
all the data with correct value.

In [ ]:
cars.reset_index(drop=True,inplace=True)

YEAR_OF_REGISTRATION

In [ ]:
cars['year_of_registration']=cars['year_of_registration'].fillna(-1)
cars['year_of_registration']=cars['year_of_registration'].astype('int64')

In [ ]:
dm={'A':1983,'B':1984,'C':1985,'CA':1985,'D':1986,'E':1987,'F':1988,'G':1989,'H':1990,'J':1991,
    'L':1993,'M':1994,'N':1995,'P':1996,'R':1997,'S':1998,'T':1999,'V':2000,'Y':2001}  #Dictionary 

In [ ]:
for i in range(len(cars)):
    if (cars.loc[i, "year_of_registration"])==-1 and pd.isna(cars.loc[i, "reg_code"]) and cars.loc[i, "mileage"]==0:
        cars.loc[i, 'year_of_registration']=2020
        cars.loc[i,'reg_code']='NEW'
    if (cars.loc[i, "year_of_registration"])==-1 and not pd.isna(cars.loc[i, "reg_code"]):
        if cars.loc[i, 'reg_code'] in dm:
            cars.loc[i, 'year_of_registration']=dm[cars.loc[i, 'reg_code']]   #Noise
    if (cars.loc[i, "year_of_registration"])==-1 and (cars.loc[i, "reg_code"]).isalnum():
         cars.loc[i, 'year_of_registration']=cars['year_of_registration'].median()

In [ ]:
outliers=cars.query('0<=year_of_registration<=2006')
cars.drop(outliers.index,inplace=True)
cars.reset_index(drop=True,inplace=True)

In [ ]:
cars['year_of_registration'].unique()

In [ ]:
cars = cars.dropna(subset=["reg_code"])
cars.reset_index(drop=True,inplace=True)

## 1.4 Encoding categorical features

In [ ]:
cars_v1 = cars.copy()

In [ ]:
cars_v1['age'] = 2022 - cars_v1['year_of_registration']  # age of the car
cars_v1['mileage_per_year'] = cars_v1['mileage'] / cars_v1['age'] # calculating mileage per year
cars_v1['mileage_per_year'] = cars_v1['mileage_per_year'].round(2) # rounding off the floating value with upto 2 decimal point

In [ ]:
cars_v1

In [ ]:
# pip install category_encoders (Just in case needed)

# Target encoding

import category_encoders as ce

# define the target encoding and fit on the data
target_encoder = ce.TargetEncoder(cols=['standard_colour', 'standard_make', 'standard_model', 'vehicle_condition', 'body_type','fuel_type', 'crossover_car_and_van','year_of_registration','age','mileage_per_year'])
encoded_df = target_encoder.fit_transform(cars_v1, cars_v1['price'])

In [ ]:
# Label encoder

le = LabelEncoder()

columns_to_encode = ['standard_make', 'standard_model', 'standard_colour', 'body_type', 'fuel_type','reg_code','vehicle_condition','crossover_car_and_van']

for col in columns_to_encode:
    le.fit(cars_v1[col])
    cars_v1[col] = le.transform(cars_v1[col])

In [ ]:
cars_v1 = encoded_df.copy()

In [ ]:
cars_v1

## 1.5 Rescaling & Splitting the data

In [ ]:
# Rescaling the dataset using MinMaxScaler

from sklearn.preprocessing import MinMaxScaler

# Define the columns to rescale
num_columns = ['mileage', 'year_of_registration', 'price']

# Create a Pipeline for rescaling the data
scaler_pipeline = Pipeline([
    ('scaler', MinMaxScaler())
])

# Apply the pipeline to rescale the numerical columns
cars_v1[num_columns] = scaler_pipeline.fit_transform(cars[num_columns])

In [ ]:
# Rescaling the dataset using StandardScaler

from sklearn.preprocessing import StandardScaler

# Define the columns to rescale
num_columns = ['mileage', 'year_of_registration', 'price']

# Create a Pipeline for rescaling the data
rescale_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

# Apply the pipeline to rescale the numerical columns
cars_v1[num_columns] = rescale_pipeline.fit_transform(cars_v1[num_columns])

In [ ]:
# Sampling the dataset and splitting into predictors & target using pipeline

from sklearn.preprocessing import FunctionTransformer

top_brands = cars['standard_make'].value_counts().nlargest(15).index

pipeline = Pipeline([
    ('brand_filter', FunctionTransformer(lambda x: x[x['standard_make'].isin(top_brands)])),
    ('sample', FunctionTransformer(lambda x: x.sample(frac=0.1, replace=True))),
    ('split', FunctionTransformer(lambda x: (x.drop(columns=['price']), x['price'])))
])

X, y = pipeline.fit_transform(cars_v1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Sampling the dataset with specfic condition from top brand cars and 20% from original dataset

top_brands = cars_v1['standard_make'].value_counts().nlargest(15).index
top_brands_df = cars_v1[cars_v1['standard_make'].isin(top_brands)]
top_brands_df = top_brands_df.sample(frac=0.2,replace=True)

In [ ]:
cars_v1 = top_brands_df

# 2. Feature Engineering

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge

In [ ]:
cars_v2 = cars_v1.copy()

In [ ]:
cars_v2

### 2.1 Deriving new features

In [ ]:
cars_v2['age'] = 2022 - cars_v2['year_of_registration']  # age of the car
cars_v2['mileage_per_year'] = cars_v2['mileage'] / cars_v2['age'] # calculating mileage per year
cars_v2['mileage_per_year'] = cars_v2['mileage_per_year'].round(2) # rounding off the floating value 

In [ ]:
# Dropping features that are not likely to be useful for predicting the target variable 
cars_v22 = cars_v2.copy()

cars_v22.drop(['public_reference', 'reg_code'], axis=1, inplace=True)

### 2.2 Polynomial Features

In [ ]:
# Define the predictors and target
X = cars_v22.drop(columns=['price'])
y = cars_v22['price']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the pipeline without polynomial features
pipe_no_poly = Pipeline(steps=[
    ('est', LinearRegression())
])

# Fit the pipeline without polynomial features on the training data
pipe_no_poly.fit(X_train, y_train)

# Predict on the test set without polynomial features
y_pred_no_poly = pipe_no_poly.predict(X_test)

# Calculate RMSE without polynomial features
rmse_no_poly = mean_squared_error(y_test, y_pred_no_poly, squared=False)
print("RMSE without polynomial features:", rmse_no_poly)

# Define the pipeline with polynomial features
pipe = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('est', LinearRegression())
])

# Fit the pipeline with polynomial features on the training data
pipe.fit(X_train, y_train)

# Predict on the test set with polynomial features
y_pred_poly = pipe.predict(X_test)

# Calculate RMSE with polynomial features
rmse_poly = mean_squared_error(y_test, y_pred_poly, squared=False)
print("RMSE with polynomial features:", rmse_poly)

### 2.3 Interactions

In [ ]:
# create a pipeline without interactions
pipe_no_interaction = Pipeline(
    steps=[
        ('poly', PolynomialFeatures(include_bias=False)),
        ('est', LinearRegression())
    ]
)

# fit the pipeline on the training data
pipe_no_interaction.fit(X_train, y_train)

# predict the target values for the test data
y_pred_no_interaction = pipe_no_interaction.predict(X_test)

# calculate the RMSE score for the predictions without interactions
rmse_no_interaction = mean_squared_error(y_test, y_pred_no_interaction, squared=False)

print('RMSE without interactions:', rmse_no_interaction)

# Pipeline with polynomial features and Ridge regression
interaction = Pipeline(steps=[
    ('poly', PolynomialFeatures(interaction_only=True, include_bias=False))
]).set_output(transform="pandas")

pipe_interaction = Pipeline(
    steps=[
        ('pp', interaction),
        ('est', LinearRegression())
    ]
)

# Fit the pipeline to the training data
pipe_interaction.fit(X_train, y_train)

# Make predictions on the testing data
y_pred = pipe_interaction.predict(X_test)
\
# Calculate RMSE
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f"RMSE with polynomial features and interactions: {rmse:.2f}")

# 3. Feature Selection

In [ ]:
cars_v3 = cars_v2.copy()

In [ ]:
cars_v3

In [ ]:
corr_matrix = cars_v3.corr()

# plot the correlation matrix
plt.figure(figsize=(10, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

In [ ]:
X = cars_v3.drop('price', axis=1) # predictors
y = cars_v3['price'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

### Manual Selection

In [ ]:
X1 = X[['mileage', 'standard_make', 'standard_colour','standard_model','body_type','year_of_registration','vehicle_condition','mileage_per_year']]

In [ ]:
X_train_man, X_test_man, y_train_man, y_test_man = train_test_split(X1, y, test_size=1/4, random_state=0)
X_train_man.shape, X_test_man.shape, y_train_man.shape, y_test_man.shape

In [ ]:
def calculate_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

model = Ridge().fit(X_train_man, y_train_man)
calculate_rmse(y_test_man, model.predict(X_test_man))

###  3.1 Automated Feature Selection (AFS)

In [ ]:
cars_v3.drop(['public_reference', 'reg_code'], axis=1, inplace=True)

In [ ]:
X = cars_v3.drop('price', axis=1) # predictors
y = cars_v3['price'] # target variable

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline, make_pipeline

In [ ]:
selector = make_pipeline(
    SelectKBest(f_regression, k=7)
).set_output(transform='pandas').fit(X, y)
X_sel = selector.transform(X)

In [ ]:
X_sel.columns

In [ ]:
model = Ridge().fit(X_sel, y)
scores = cross_val_score(model, X_sel, y)
scores.mean(), scores.std()

In [ ]:
model = Ridge().fit(X, y)
scores = cross_val_score(model, X, y)
scores.mean(), scores.std()

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Define the number of top features to select
k = 7

# Define the selector pipeline to select the top k features using f_regression
selector = make_pipeline(
    SelectKBest(f_regression, k=k)
).set_output(transform='pandas')

# Fit the selector pipeline on the training data and transform the features
X_train_sel = selector.fit_transform(X_train, y_train)

# Fit a Ridge regression model on the selected features and evaluate using cross-validation
model = Ridge()
scores_sel = cross_val_score(model, X_train_sel, y_train)
print(f"Mean cross-validation score with top {k} features: {scores_sel.mean():.3f} (+/- {scores_sel.std():.3f})")

# Fit a Ridge regression model on all the features and evaluate using cross-validation
scores_all = cross_val_score(model, X_train, y_train)
print(f"Mean cross-validation score with all features: {scores_all.mean():.3f} (+/- {scores_all.std():.3f})")

### 3.2 Recursive Feature Elimination (RFE)

In [ ]:
from sklearn.feature_selection import RFECV

In [ ]:
model = Ridge()
ref_selector = RFECV(model, step=1, cv=5)

In [ ]:
ref_selector.fit(X, y)

In [ ]:
ref_selector.get_feature_names_out()

In [ ]:
n_scores = len(ref_selector.cv_results_["mean_test_score"])
n_scores

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.errorbar(
    range(1, n_scores+1),
    ref_selector.cv_results_["mean_test_score"],
    yerr=ref_selector.cv_results_["std_test_score"],
)
ax.set_xlabel("Number of features selected")
ax.set_ylabel("Mean test score");

### 3.3 Sequential Feature Selection (SFS) (Forward/Backward)

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector

In [ ]:
sfs_forward = SequentialFeatureSelector(
    Ridge(), n_features_to_select=7, direction="forward"
).fit(X, y)

In [ ]:
sfs_forward.get_feature_names_out()

### 3.4 Dimensionality Reduction

In [ ]:
cars_v3

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA()
pca_full.fit(cars_v3)

In [ ]:
pca_full.explained_variance_ratio_

In [ ]:
plt.plot(np.cumsum(pca_full.explained_variance_ratio_))
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance');

In [ ]:
pca = PCA(n_components=2)
pca.fit(cars_v3)
X_pca = pca.transform(cars_v3)
print("Original shape: {}".format(str(cars_v3.shape)))
print("Reduced shape: {}".format(str(X_pca.shape)))

In [ ]:
c1_df = pd.Series(pca.components_[0], index=cars_v3.columns)
c1_df.abs().sort_values(ascending=True)

In [ ]:
c1_df = pd.Series(pca.components_[1], index=cars_v3.columns)
c1_df.abs().sort_values(ascending=True)

Selected features from the above all alogithms : 
<br>PCA : mileage, mileage_per_year, standard_model, age, year_of_registration, public_reference, body_type, reg_code
<br>SelectKBest: mileage, mileage_per_year, age, year_of_registration, vehicle_condition
<br>Sequential Feature: mileage_per_year, age, standard_make, standard_model, vehicle_condition, body_type , fuel_type 
<br>Recursive Feature : mileage, reg_code, standard_colour, standard_make, standard_model, vehicle_condition, year_of_registration, body_type, crossover_car_and_van, fuel_type, age, mileage_per_year, public_reference

Further selected features: mileage, mileage_per_year, standard_make, standard_model, year_of_registration, age, body_type, fuel_type, standard_colour

# 4. Model Building

In [ ]:
cars_v4 = cars_v3.copy()

In [ ]:
cars_v4.drop(['vehicle_condition','crossover_car_and_van'], axis=1, inplace=True)

In [ ]:
cars_v4

In [ ]:
X, y = cars_v4.drop(columns='price'), cars_v4['price']
# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

### 4.1 Liner Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
rmse_linear = mean_squared_error(y_test, y_pred, squared=False)

print("RMSE with Linear Regressor:", rmse_linear)

### 4.2 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

rf_rmse = mean_squared_error(y_test, rf_model.predict(X_test), squared=False)
print("RMSE for Random Forest:", rf_rmse)

### Ridge & Lasso Regressor

In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Ridge regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
rmse_ridge = mean_squared_error(y_test, y_pred_ridge, squared=False)
print("RMSE for Ridge regression:", rmse_ridge)

# Lasso regression
lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
rmse_lasso = mean_squared_error(y_test, y_pred_lasso, squared=False)
print("RMSE for Lasso regression:", rmse_lasso)

### 4.3 Boosted Tree

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

bt_model = GradientBoostingRegressor(random_state=42)
bt_model.fit(X_train, y_train)
y_pred = bt_model.predict(X_test)

# Calculate the root-mean-square error of the model
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print('RMSE for GradientBoostingRegressor:', rmse)

In [ ]:
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor

ht_model = HistGradientBoostingRegressor().fit(X_train, y_train)
y_pred = ht_model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)

print("RMSE for HistGradientBoostingRegressor:", rmse)

### Model Building with Polynomial Features

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)

In [ ]:
lr_poly = LinearRegression()
lr_poly.fit(X_train_poly, y_train)
y_pred_poly = lr_poly.predict(X_test_poly)
rmse_linear_poly = mean_squared_error(y_test, y_pred_poly, squared=False)

print("RMSE for Linear Regressor with Polynomial Features:", rmse_linear_poly)

In [ ]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_poly = poly.fit_transform(X)

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(X_poly, y, test_size=0.2, random_state=42)

rf_model_poly = RandomForestRegressor(random_state=42)
rf_model_poly.fit(X_train_poly, y_train_poly)

# Evaluate the Random Forest Regressor model with transformed data
rf_rmse_poly = mean_squared_error(y_test_poly, rf_model_poly.predict(X_test_poly), squared=False)
print("RMSE for Random Forest with polynomial features:", rf_rmse_poly)

In [ ]:
pipe_poly = Pipeline([
    ('poly', PolynomialFeatures(interaction_only=True, include_bias=False)),
    ('model', GradientBoostingRegressor(random_state=42))
])

# Fit the model and calculate the RMSE score with polynomial features
pipe_poly.fit(X_train, y_train)
rmse_poly = mean_squared_error(y_test, pipe_poly.predict(X_test), squared=False)
print("RMSE for GradientBoostingRegressor with polynomial features: ", rmse_poly)

In [ ]:
pipe = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("estimator", HistGradientBoostingRegressor(random_state=42))
]).set_output(transform="pandas")

# transform the data with polynomial features
X_train_poly = pipe["poly"].fit_transform(X_train)
X_test_poly = pipe["poly"].transform(X_test)

# fit the model with transformed data
pipe.fit(X_train_poly, y_train)

# calculate the RMSE with transformed data
rmse_poly = mean_squared_error(y_test, pipe.predict(X_test_poly), squared=False)
print("RMSE for HistGradientBoostingRegressor with polynomial features:", rmse_poly)

## 4.4 Model Ranking

Linear Regressor without Polynomial features 	5866.792749
<br>Linear Regressor with Polynomial Features	5130.062002
<br>Random Forest without polynomial features	3437.586276
<br>Random Forest with polynomial features 	3487.600364
<br>RMSE for Ridge regression	5866.792749
<br>RMSE for Lasso regression	5866.792793
<br>GradientBoostingRegressor without Polynomial features	4467.978783
<br>GradientBoostingRegressor with polynomial features 	4375.696558
<br>HistGradientBoostingRegressor without Polynomial features	3947.123108
<br>HistGradientBoostingRegressor with polynomial features 	3642.419985

## 4.5 Grid Search for top 3 rank models: 

### 4.5.1 Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import PolynomialFeatures

pipeline = Pipeline(steps=[('poly', PolynomialFeatures()),
                           ('rf', RandomForestRegressor())])

param_grid = {'poly__degree': [1, 2],
              'rf__n_estimators': [50, 100],
              'rf__max_depth': [None, 5, 10]
             }

grid_search = GridSearchCV(pipeline, param_grid=param_grid, cv=5, n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

print("Best parameters: ", grid_search.best_params_)

In [ ]:
# define the pipeline with polynomial features and random forest regressor
rf_pipeline = make_pipeline(PolynomialFeatures(degree=1), 
                             RandomForestRegressor(max_depth=None, n_estimators=100, random_state=42))

rf_pipeline.fit(X_train, y_train)

y_pred = rf_pipeline.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)

print("RMSE score:", rmse)

### 4.5.2 Gradient Boosting 

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

gb = GradientBoostingRegressor()

param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.05, 0.1, 0.2]
}

grid_search = GridSearchCV(gb, param_grid=param_grid, cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_

y_pred = grid_search.predict(X_test)

rmse_score = mean_squared_error(y_test, y_pred, squared=False)

# print best parameters, rmse score
print("Best parameters:", best_params)
print("RMSE score:", rmse_score)

### 4.5.3 Hist Grading

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

model = HistGradientBoostingRegressor()

param_grid = {'learning_rate': [0.1, 0.05, 0.01],
              'max_iter': [100, 200, 300],
              'max_depth': [None, 5, 10]}

grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, n_jobs=-1)
grid_result = grid.fit(X_train, y_train)

print("Best Parameters: ", grid_result.best_params_)

y_pred = grid_result.best_estimator_.predict(X_test)

print("RMSE: ", np.sqrt(mean_squared_error(y_test, y_pred)))

## 4.6 Ensemble

In [ ]:
from sklearn.ensemble import VotingRegressor

# Define the models with their best configurations
model1 = make_pipeline(PolynomialFeatures(degree=2), RandomForestRegressor(max_depth=None, n_estimators=100))
model2 = GradientBoostingRegressor(n_estimators=150, learning_rate=0.2, max_depth=4)
model3 = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.1, max_depth=10)

# Build the ensemble with the best models
ensemble = VotingRegressor([('rf', model1), ('gb', model2), ('hgb', model3)])

# Fit the ensemble on the data
ensemble.fit(X_train, y_train)

# Make predictions on the test set
y_pred = ensemble.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print('RMSE:', rmse)

## 5 Model Evaluation and Analysis

### 5.1 Overall Performance with Cross-Validation

In [ ]:
# Define the models
model1 = RandomForestRegressor()
model2 = GradientBoostingRegressor()
model3 = HistGradientBoostingRegressor()
ensemble = VotingRegressor([('rf', model1), ('gb', model2), ('hgb', model3)])

# Cross-validation
cv_scores = {}
for model in [model1, model2, model3, ensemble]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    cv_scores[type(model).__name__] = np.sqrt(-scores)
    
# Calculate mean and standard deviation of cross-validation scores
for name, scores in cv_scores.items():
    print(f"{name} - Mean : {scores.mean():.2f}, Standard Deviation: {scores.std():.2f}")

### 5.2 True vs Predicted Analysis

In [ ]:
import matplotlib.pyplot as plt

rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Plot the true vs predicted values
fig, ax = plt.subplots()
ax.scatter(y_test, y_pred, edgecolors=(0, 0, 0))
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3)
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted') 
ax.set_title('True Vs Predicted plot for RandomForest')
plt.show()

In [ ]:
ensemble.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# Plot the true vs predicted values
fig, ax = plt.subplots()
ax.scatter(y_test, y_pred, edgecolors=(0, 0, 0))
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=3)
ax.set_xlabel('Actual')
ax.set_ylabel('Predicted') 
ax.set_title('True Vs Predicted plot for Ensemble')
plt.show()

### 5.3 Global and Local Explanations with SHAP

In [ ]:
cars_sample = cars_v4.sample(n=2000, random_state=42)

In [ ]:
# Split the dataset 
X_sample = cars_sample.drop(['price'], axis=1)
y_sample = cars_sample['price']

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, random_state=42)

In [ ]:
import shap #pip install shap (just in cases needed)

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0,:], X_test.iloc[0,:])

In [ ]:
shap.summary_plot(shap_values, X_test, max_display=8)

### 5.4 Individual Conditional Expectation

Let us investigate the effect of mileage across the whole range

In [ ]:
X_sample['mileage'].min(), X_sample['mileage'].max()

In [ ]:
mil_synth_data = np.linspace(X_sample['mileage'].min(), X_sample['mileage'].max(), 100)
mil_synth_data[:5], mil_synth_data[-5:]

In [ ]:
ex_instance = X_sample.sample(1, random_state=42).drop(columns='mileage')
ex_instance

In [ ]:
mil_synth_df = pd.DataFrame(mil_synth_data, columns=['mileage'])
mil_synth_df.head()

In [ ]:
synth_df = ex_instance.merge(mil_synth_df, how='cross')
print(synth_df.shape)
synth_df

In [ ]:
synth_df = synth_df[X_train.columns]  # Reorder the columns to match the order in X_train

pred = rf.predict(synth_df)
pred[:5], pred[-5:]

In [ ]:
sns.displot(X_sample['mileage'], kde=True, rug=True);

In [ ]:
ax = sns.lineplot(x=synth_df['mileage'], y=pred);

### 5.5 Partial Dependency Plot

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

In [ ]:
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=['mileage'], kind='both'
);

In [ ]:
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)
PartialDependenceDisplay.from_estimator(
    rf, X_test, features=['standard_model','mileage_per_year','age','standard_colour'],
    kind='both',
    subsample=100, grid_resolution=30, n_jobs=2, random_state=0,
    ax=ax, n_cols=2
);